In [13]:
import time
import warnings
from collections import Counter
from itertools import combinations

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

In [11]:
df_train = pd.read_csv("train_processed.csv")
df_val = pd.read_csv("val_processed.csv")

In [17]:
categorical_features = ["itemId", "modelId", "cat_id", "brand", "promotionId"]
price_features = ["priceBeforeDiscount", "item_price_min", "item_price_max"]

In [18]:
def prepare_data(
    df_train,
    df_val,
    categorical_features,
    price_features,
    cat_feature,
):
    # Drop all categorical features except the selected one
    drop_cols = [c for c in categorical_features if c != cat_feature]

    train_processed = df_train.drop(columns=drop_cols).copy()
    val_processed = df_val.drop(columns=drop_cols).copy()

    # Log transform price-related features
    train_processed[price_features] = np.log1p(train_processed[price_features])
    val_processed[price_features] = np.log1p(val_processed[price_features])

    # Convert selected categorical feature
    train_processed[cat_feature] = train_processed[cat_feature].astype("category")
    val_processed[cat_feature] = val_processed[cat_feature].astype("category")

    # Split features and target
    X_train = train_processed.drop(columns=["price"])
    X_val = val_processed.drop(columns=["price"])

    y_train = train_processed["price"]
    y_val = val_processed["price"]

    # Log transform target
    y_train_log = np.log1p(y_train)
    y_val_log = np.log1p(y_val)

    return (
        X_train,
        X_val,
        y_train,
        y_val,
        y_train_log,
        y_val_log,
    )


def eval_model(
    cat_feature,
    X_train,
    y_val,
    y_pred,
    train_time,
    total_time,
):
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mape = mean_absolute_percentage_error(y_val, y_pred) * 100
    r2 = r2_score(y_val, y_pred)

    result = {
        "categorical_feature": cat_feature,
        "n_features": X_train.shape[1],
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "train_time_sec": train_time,
        "total_time_sec": total_time,
    }

    print(f"MAE        : {mae:.4f}")
    print(f"RMSE       : {rmse:.4f}")
    print(f"MAPE       : {mape:.2f}%")
    print(f"R²         : {r2:.4f}")
    print(f"Train Time : {train_time:.2f} sec")
    print(f"Total Time : {total_time:.2f} sec")

    return result

### XGBoost

In [19]:
results = []

for cat_feature in categorical_features:

    print(f"\n{'='*60}")
    print(f"Using categorical feature: {cat_feature}")
    print(f"{'='*60}")

    (
        X_train,
        X_val,
        y_train,
        y_val,
        y_train_log,
        y_val_log,
    ) = prepare_data(
        df_train=df_train,
        df_val=df_val,
        categorical_features=categorical_features,
        price_features=price_features,
        cat_feature=cat_feature,
    )

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        enable_categorical=True,
        random_state=42,
    )

    total_start = time.perf_counter()

    train_start = time.perf_counter()

    model.fit(
        X_train,
        y_train_log,
        eval_set=[(X_val, y_val_log)],
        verbose=False,
    )

    train_time = time.perf_counter() - train_start

    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)

    total_time = time.perf_counter() - total_start

    results.append(
        eval_model(
            cat_feature=cat_feature,
            X_train=X_train,
            y_val=y_val,
            y_pred=y_pred,
            train_time=train_time,
            total_time=total_time,
        )
    )


Using categorical feature: itemId
MAE        : 25.9771
RMSE       : 99.7723
MAPE       : 7.72%
R²         : 0.9885
Train Time : 13.39 sec
Total Time : 13.59 sec

Using categorical feature: modelId
MAE        : 1.9201
RMSE       : 21.0499
MAPE       : 0.41%
R²         : 0.9995
Train Time : 25.15 sec
Total Time : 25.31 sec

Using categorical feature: cat_id
MAE        : 28.9150
RMSE       : 118.3115
MAPE       : 7.82%
R²         : 0.9838
Train Time : 7.43 sec
Total Time : 7.52 sec

Using categorical feature: brand
MAE        : 28.6927
RMSE       : 126.1115
MAPE       : 7.84%
R²         : 0.9816
Train Time : 9.62 sec
Total Time : 9.77 sec

Using categorical feature: promotionId
MAE        : 31.2592
RMSE       : 146.7088
MAPE       : 7.93%
R²         : 0.9752
Train Time : 7.66 sec
Total Time : 7.78 sec


In [20]:
# Create a summary table
results_df = pd.DataFrame(results)

results_df = results_df.sort_values("RMSE").reset_index(drop=True)

In [21]:
results_df

,categorical_feature,n_features,MAE,RMSE,MAPE,R2,train_time_sec,total_time_sec
0,modelId,19,1.920103,21.049868,0.407234,0.999489,25.149876,25.308040
1,itemId,19,25.977091,99.772325,7.718277,0.988512,13.389863,13.590858
2,cat_id,19,28.914961,118.311458,7.821882,0.983846,7.431065,7.515977
3,brand,19,28.692653,126.111526,7.839050,0.981646,9.617021,9.766679
4,promotionId,19,31.259177,146.708848,7.928999,0.975160,7.663474,7.775230


### CatBoost

In [22]:
from catboost import CatBoostRegressor

In [23]:
results = []

for cat_feature in categorical_features:

    print(f"\n{'='*60}")
    print(f"Using categorical feature: {cat_feature}")
    print(f"{'='*60}")

    (
        X_train,
        X_val,
        y_train,
        y_val,
        y_train_log,
        y_val_log,
    ) = prepare_data(
        df_train=df_train,
        df_val=df_val,
        categorical_features=categorical_features,
        price_features=price_features,
        cat_feature=cat_feature,
    )

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
    )

    total_start = time.perf_counter()

    train_start = time.perf_counter()

    model.fit(
        X_train,
        y_train_log,
        cat_features=[X_train.columns.get_loc(cat_feature)],
        eval_set=(X_val, y_val_log),
        use_best_model=True,
    )

    train_time = time.perf_counter() - train_start

    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)

    total_time = time.perf_counter() - total_start

    results.append(
        eval_model(
            cat_feature=cat_feature,
            X_train=X_train,
            y_val=y_val,
            y_pred=y_pred,
            train_time=train_time,
            total_time=total_time,
        )
    )


Using categorical feature: itemId
MAE        : 30.5511
RMSE       : 108.1805
MAPE       : 8.47%
R²         : 0.9865
Train Time : 31.10 sec
Total Time : 31.11 sec

Using categorical feature: modelId
MAE        : 25.8292
RMSE       : 91.9422
MAPE       : 6.65%
R²         : 0.9902
Train Time : 30.48 sec
Total Time : 30.49 sec

Using categorical feature: cat_id
MAE        : 30.6219
RMSE       : 108.3193
MAPE       : 8.46%
R²         : 0.9865
Train Time : 29.77 sec
Total Time : 29.78 sec

Using categorical feature: brand
MAE        : 30.5972
RMSE       : 107.7572
MAPE       : 8.51%
R²         : 0.9866
Train Time : 29.94 sec
Total Time : 29.95 sec

Using categorical feature: promotionId
MAE        : 30.4945
RMSE       : 108.2140
MAPE       : 8.47%
R²         : 0.9865
Train Time : 32.63 sec
Total Time : 32.64 sec


In [24]:
results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
results_df

,categorical_feature,n_features,MAE,RMSE,MAPE,R2,train_time_sec,total_time_sec
0,modelId,19,25.829240,91.942241,6.647132,0.990244,30.478310,30.494630
1,promotionId,19,30.494458,108.213952,8.468433,0.986486,32.627769,32.642444
2,itemId,19,30.551116,108.180525,8.471937,0.986494,31.098718,31.105938
3,brand,19,30.597226,107.757240,8.510929,0.986599,29.944944,29.953349
4,cat_id,19,30.621858,108.319323,8.460370,0.986459,29.772880,29.783046


### LightBGM

In [25]:
import lightgbm as lgb
import time
import numpy as np

In [26]:
results = []

for cat_feature in categorical_features:

    print(f"\n{'='*60}")
    print(f"Using categorical feature: {cat_feature}")
    print(f"{'='*60}")

    (
        X_train,
        X_val,
        y_train,
        y_val,
        y_train_log,
        y_val_log,
    ) = prepare_data(
        df_train=df_train,
        df_val=df_val,
        categorical_features=categorical_features,
        price_features=price_features,
        cat_feature=cat_feature,
    )

    # Make sure the categorical feature has dtype 'category'
    X_train[cat_feature] = X_train[cat_feature].astype("category")
    X_val[cat_feature] = X_val[cat_feature].astype("category")

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=-1,
    )

    total_start = time.perf_counter()

    train_start = time.perf_counter()

    model.fit(
        X_train,
        y_train_log,
        eval_set=[(X_val, y_val_log)],
        categorical_feature=[cat_feature],
    )

    train_time = time.perf_counter() - train_start

    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)

    total_time = time.perf_counter() - total_start

    results.append(
        eval_model(
            cat_feature=cat_feature,
            X_train=X_train,
            y_val=y_val,
            y_pred=y_pred,
            train_time=train_time,
            total_time=total_time,
        )
    )


Using categorical feature: itemId
MAE        : 26.4937
RMSE       : 102.2200
MAPE       : 7.56%
R²         : 0.9879
Train Time : 4.54 sec
Total Time : 4.83 sec

Using categorical feature: modelId
MAE        : 7.0364
RMSE       : 53.5180
MAPE       : 1.30%
R²         : 0.9967
Train Time : 6.64 sec
Total Time : 6.92 sec

Using categorical feature: cat_id
MAE        : 27.6838
RMSE       : 100.8711
MAPE       : 8.02%
R²         : 0.9883
Train Time : 4.24 sec
Total Time : 4.45 sec

Using categorical feature: brand
MAE        : 27.4008
RMSE       : 101.0275
MAPE       : 8.00%
R²         : 0.9882
Train Time : 5.92 sec
Total Time : 6.14 sec

Using categorical feature: promotionId
MAE        : 27.7964
RMSE       : 100.6676
MAPE       : 8.11%
R²         : 0.9883
Train Time : 3.77 sec
Total Time : 3.99 sec


In [27]:
results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
results_df

,categorical_feature,n_features,MAE,RMSE,MAPE,R2,train_time_sec,total_time_sec
0,modelId,19,7.036377,53.517957,1.298867,0.996695,6.639249,6.918534
1,itemId,19,26.493669,102.219985,7.556895,0.987941,4.544478,4.833594
2,brand,19,27.400802,101.027502,7.996289,0.988221,5.915854,6.139043
3,cat_id,19,27.683808,100.871125,8.018376,0.988257,4.236506,4.445590
4,promotionId,19,27.796425,100.667574,8.110409,0.988305,3.767289,3.988444
